In [ ]:
from pathlib import Path
import zipfile

import kagglehub
import pandas as pd
import plotly.express as px

# Create data directory
data_dir = Path("../data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

# Download dataset
dataset_path = Path(
    kagglehub.dataset_download("excel4soccer/espn-soccer-data")
)

print("Dataset downloaded to:", dataset_path)

# Copy all files to data/
for source in dataset_path.rglob("*"):

    if source.is_file():

        # Preserve the directory structure
        relative_path = source.relative_to(dataset_path)
        destination = data_dir / relative_path

        # Create necessary directories
        destination.parent.mkdir(parents=True, exist_ok=True)

        # Copy file
        destination.write_bytes(source.read_bytes())

# Extract ZIP files inside data/
for zip_path in data_dir.rglob("*.zip"):

    out_dir = zip_path.with_suffix("")

    if not out_dir.exists():
        out_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(out_dir)

print("Ready.")
print("Data saved to:", data_dir.resolve())

In [ ]:
from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../data/raw/commentary_data/commentary_2025_ENG.1.csv")
output_csv = Path("../data/processed/penalties.csv")

df = pd.read_csv(input_csv)


def extract_penalty(row):
    text = str(row["commentaryText"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })

    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"top (centre|center)", text):
        position, direction = 2, "top center"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"(centre|center) right", text):
        position, direction = 4, "middle right"
    elif re.search(
        r"(centre|center) of the goal|down the middle|middle of the goal",
        text
    ):
        position, direction = 5, "middle center"
    elif re.search(r"(centre|center) left", text):
        position, direction = 6, "middle left"

    elif re.search(r"bottom right", text):
        position, direction = 7, "bottom right"
    elif re.search(r"bottom (centre|center)", text):
        position, direction = 8, "bottom center"
    elif re.search(r"bottom left", text):
        position, direction = 9, "bottom left"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    elif re.search(r"to the right|misses to the right", text):
        position, direction = 4, "middle right"
    elif re.search(r"to the left|misses to the left", text):
        position, direction = 6, "middle left"
    else:
        position, direction = pd.NA, pd.NA

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
output_csv.parent.mkdir(parents=True, exist_ok=True)
penalties.to_csv(output_csv, index=False)

print(penalties[
    [
        "eventId",
        "commentaryOrder",
        "commentaryText",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv.resolve())